# 당뇨병 데이터 분석 — 답지용(solution)

scikit-learn 내장 당뇨병(diabetes) 데이터셋으로 수행하는 탐색적 분석(EDA)과 회귀 모델링.

셀을 위에서부터 순서대로 실행하며 결과를 눈으로 확인.

> 모든 설명 문장은 명사형 어미로 작성.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 변수 한글 설명 (피처명 → 의미)
FEATURE_DESC = {
    "age": "나이", "sex": "성별", "bmi": "체질량지수", "bp": "평균 혈압",
    "s1": "총 콜레스테롤(TC)", "s2": "LDL", "s3": "HDL", "s4": "TC/HDL 비율",
    "s5": "혈청 중성지방 로그값(ltg)", "s6": "혈당(glu)",
}
print("라이브러리 로드 완료")

## 1. 데이터 로드 및 개요

442명 환자의 10개 지표와 1년 후 진행도(target)로 구성된 데이터셋 확인.

In [ ]:
# 데이터 로드 후 DataFrame 구성
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")
df = X.copy()
df["target"] = y
print("형태:", df.shape)
df.head()

### 변수 설명

In [ ]:
# 변수 설명 테이블
pd.DataFrame({"변수": list(FEATURE_DESC.keys()), "의미": list(FEATURE_DESC.values())})

### 요약 통계

In [ ]:
# 요약 통계
df.describe().round(3)

## 2. 탐색적 분석(EDA)

### 2.1 타깃 분포

타깃 변수의 분포 형태 파악.

In [ ]:
# 타깃 분포 히스토그램 (상자그림 동반)
fig = px.histogram(df, x="target", nbins=30, marginal="box",
                   title="1년 후 질병 진행도 분포")
fig.show()
print(f"평균 {y.mean():.1f} / 중앙값 {y.median():.1f} / 표준편차 {y.std():.1f}")

### 2.2 상관관계

피처 간 및 타깃과의 상관 구조 파악.

In [ ]:
# 상관관계 히트맵
corr = df.corr()
fig = px.imshow(corr.round(2), text_auto=True, aspect="auto",
                color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="피처 간 상관관계")
fig.show()

In [ ]:
# 타깃과의 상관계수 (내림차순)
corr_t = corr["target"].drop("target").sort_values(ascending=False)
pd.DataFrame({"변수": [f"{k} ({FEATURE_DESC[k]})" for k in corr_t.index],
              "상관계수": corr_t.values.round(3)})

### 2.3 상위 변수 산점도

타깃과 상관이 높은 변수의 관계 시각화.

In [ ]:
# 타깃과 상관이 높은 상위 변수의 산점도 + 추세선
top_feats = corr["target"].drop("target").abs().sort_values(ascending=False).head(4).index.tolist()
for f in top_feats:
    fig = px.scatter(df, x=f, y="target", trendline="ols", opacity=0.5,
                     title=f"{f} ({FEATURE_DESC[f]}) vs target  r={corr.loc[f,'target']:.2f}")
    fig.show()

## 3. 회귀 모델링

### 3.1 데이터 분할

In [ ]:
# 학습 80% / 테스트 20% 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print("학습:", X_train.shape[0], "/ 테스트:", X_test.shape[0])

### 3.2 모델 학습 및 성능 비교

5종 모델 학습 후 테스트 성능과 교차검증 결과 비교.

In [ ]:
# 5종 모델 학습 및 5-fold 교차검증 평가
models = {
    "Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3, 1, 50), max_iter=10000)),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows, fitted = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv_r2 = cross_val_score(model, X, y, cv=cv, scoring="r2")
    rows.append({"모델": name,
                 "Test R²": r2_score(y_test, pred),
                 "CV R² 평균": cv_r2.mean(),
                 "CV R² 표준편차": cv_r2.std(),
                 "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
                 "MAE": mean_absolute_error(y_test, pred)})
    fitted[name] = model

perf = pd.DataFrame(rows).sort_values("Test R²", ascending=False).reset_index(drop=True)
perf.round(3)

In [ ]:
# 모델별 R² 비교 막대그래프
perf_melt = perf.melt(id_vars="모델", value_vars=["Test R²", "CV R² 평균"],
                      var_name="지표", value_name="R²")
fig = px.bar(perf_melt, x="모델", y="R²", color="지표", barmode="group",
             title="모델별 R² 비교")
fig.show()

### 3.3 변수 중요도

Ridge 계수와 RandomForest 중요도 비교 — bmi와 s5의 영향력 확인.

In [ ]:
# 변수 중요도: Ridge 계수 vs RandomForest 중요도
ridge = fitted["Ridge"].named_steps["ridgecv"]
ridge_coef = pd.Series(ridge.coef_, index=X.columns).sort_values()
px.bar(x=ridge_coef.values, y=ridge_coef.index, orientation="h",
       title="Ridge 회귀 계수", labels={"x": "계수", "y": "변수"}).show()

rf_imp = pd.Series(fitted["RandomForest"].feature_importances_, index=X.columns).sort_values()
px.bar(x=rf_imp.values, y=rf_imp.index, orientation="h",
       title="RandomForest 변수 중요도", labels={"x": "중요도", "y": "변수"}).show()

### 3.4 예측 대 실제

최고 모델의 예측 정확도 시각화.

In [ ]:
# 최고 모델의 예측 대 실제 비교
best_name = perf.iloc[0]["모델"]
best_pred = fitted[best_name].predict(X_test)
cmp = pd.DataFrame({"실제값": y_test.values, "예측값": best_pred})
fig = px.scatter(cmp, x="실제값", y="예측값", opacity=0.5,
                 title=f"{best_name} (Test R²={perf.iloc[0]['Test R²']:.3f})")
lim = [y_test.min(), y_test.max()]
fig.add_shape(type="line", x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
              line=dict(color="red", dash="dash"))
fig.show()
print("최고 모델:", best_name)

## 4. 결론 및 시사점

- 체질량지수(bmi)와 혈청 중성지방(s5)이 진행도와 가장 강하게 연관된 핵심 인자
- HDL 콜레스테롤(s3)은 음의 상관 — 보호 인자로 해석됨
- 정규화 선형 모델(Lasso/Ridge)이 트리 앙상블보다 우수 — 데이터가 선형적이기 때문
- 최고 모델도 분산의 약 47%만 설명 — 추가 피처·비선형 항 고려 시 개선 여지

> 본 분석은 교육용 데이터셋 기반이며 실제 임상 진단에는 사용 불가.